# piglot: Derivative-Free Optimization

## The Core Question

You have a CFD simulation that takes 4 hours per run. You want to find the best airfoil shape, but you can't compute gradients (the simulation is a black box). How do you optimize?

**Derivative-free optimization** (DFO) explores the design space without needing derivatives. piglot is a Python library for Bayesian optimization and other DFO methods — it's designed to find the optimum with **as few expensive evaluations as possible**.

## Why piglot?

- **Bayesian optimization**: builds a surrogate model (Gaussian Process) of your objective, then picks the next evaluation point that balances exploration vs exploitation
- **Fewer evaluations**: for expensive simulations (CFD, FEA), 50 well-chosen points beat 5000 random ones
- **Multi-objective**: handles Pareto fronts natively
- **Parallel**: evaluates multiple points simultaneously (embarrassingly parallel)

## The Workflow

```
Define objective (expensive black box)
        |
        v
  Initial random samples (10-20 points)
        |
        v
  Fit surrogate model (Gaussian Process)
        |
        v
  Pick next point (acquisition function: EI, UCB)
        |
        v
  Evaluate objective (CFD run)
        |
        v
  Update surrogate -> repeat until budget exhausted
        |
        v
  Report: best point + confidence interval
```

## Setup


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Check if piglot is installed
try:
    import piglot
    print(f'piglot version: {piglot.__version__}')
    HAS_PIGLOT = True
except ImportError:
    print('piglot is NOT installed.')
    print('Install: pip install piglot')
    print()
    print('This notebook will demonstrate the CONCEPTS using a')
    print('manual Bayesian optimization implementation so you')
    print('can follow the logic even without the library.')
    HAS_PIGLOT = False

np.random.seed(42)

## 1. The Problem: Expensive Black-Box Optimization

**Concept**: Imagine you're optimizing a turbine blade. Each evaluation is a CFD simulation (4 hours). You have a budget of 30 evaluations. How do you use them wisely?

**Random search** wastes evaluations. **Bayesian optimization** uses what it has learned to pick the next point intelligently.

We'll use a synthetic "CFD" function (fast, but shaped like a real objective) to demonstrate.


In [ ]:
# Synthetic "CFD" objective: minimize drag
# Shaped like a real aerodynamic objective: smooth, multi-modal, expensive
def fake_cfd(x):
    """
    x: array of shape (n_params,)
    Returns: drag coefficient (to minimize)
    
    Designed to be:
    - Smooth (like real CFD)
    - Multi-modal (local minima trap random search)
    - Expensive (we pretend each call takes 4 hours)
    """
    x1, x2 = x[0], x[1]
    # Base: quadratic bowl at (0.3, 0.7)
    base = (x1 - 0.3)**2 + (x2 - 0.7)**2
    # Add a secondary local minimum at (0.8, 0.2)
    local = 0.3 * ((x1 - 0.8)**2 + (x2 - 0.2)**2)
    # Add noise-like ripples (simulates mesh dependency)
    ripple = 0.05 * np.sin(10 * x1) * np.cos(8 * x2)
    return 0.01 + base + local + ripple

# Evaluate on a grid to visualize the landscape
x1_grid = np.linspace(0, 1, 100)
x2_grid = np.linspace(0, 1, 100)
X1, X2 = np.meshgrid(x1_grid, x2_grid)
Z = np.array([[fake_cfd([x1, x2]) for x1 in x1_grid] for x2 in x2_grid])

fig, ax = plt.subplots(1, 1, figsize=(8, 6))
contour = ax.contourf(X1, X2, Z, levels=30, cmap='viridis')
ax.contour(X1, X2, Z, levels=10, colors='white', linewidths=0.5, alpha=0.5)
ax.set_xlabel('Param 1 (e.g., camber)')
ax.set_ylabel('Param 2 (e.g., thickness)')
ax.set_title('Objective Landscape (fake CFD: minimize Cd)')
plt.colorbar(contour, label='Cd')
plt.tight_layout()
plt.show()

print('Global minimum is near (0.3, 0.7) with Cd ~ 0.010')
print('Local minimum near (0.8, 0.2) traps random search')

## 2. Random Search: The Baseline

**Concept**: Random search picks points uniformly at random. It's simple, parallelizable, and surprisingly hard to beat for low-dimensional problems. But it **wastes evaluations** in regions it has already explored.


In [ ]:
def random_search(n_evals, seed=42):
    """Random search: pick n_evals random points."""
    rng = np.random.RandomState(seed)
    points = rng.uniform(0, 1, size=(n_evals, 2))
    values = np.array([fake_cfd(p) for p in points])
    best_idx = np.argmin(values)
    return points, values, points[best_idx], values[best_idx]

# Run random search with 30 evaluations
pts_rs, vals_rs, best_pt_rs, best_val_rs = random_search(30)

fig, ax = plt.subplots(1, 1, figsize=(8, 6))
contour = ax.contourf(X1, X2, Z, levels=30, cmap='viridis', alpha=0.6)
ax.scatter(pts_rs[:, 0], pts_rs[:, 1], c='red', s=50, edgecolors='black',
           linewidths=0.5, label='Random points', zorder=5)
ax.scatter(best_pt_rs[0], best_pt_rs[1], c='gold', s=200, marker='*',
           edgecolors='black', linewidths=1.5, label=f'Best: Cd={best_val_rs:.4f}', zorder=6)
ax.set_xlabel('Param 1')
ax.set_ylabel('Param 2')
ax.set_title(f'Random Search: 30 evaluations, best Cd = {best_val_rs:.4f}')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

print(f'Random search best: Cd = {best_val_rs:.4f} at {best_pt_rs}')
print('Notice: many points are clustered in uninteresting regions.')

## 3. Bayesian Optimization: The Smart Way

**Concept**: Bayesian optimization (BO) maintains a **surrogate model** (Gaussian Process) of the objective. After each evaluation, it updates the model and uses an **acquisition function** to pick the next point:

- **Exploitation**: where the model predicts a low value
- **Exploration**: where the model is uncertain (high variance)

The **Expected Improvement (EI)** acquisition function balances both:
```
EI(x) = E[max(f_best - f(x), 0)]
```

We'll implement a simplified BO loop (the same logic piglot uses internally).


In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern

def expected_improvement(X, gp, y_best, xi=0.01):
    """Compute Expected Improvement acquisition function."""
    mu, sigma = gp.predict(X, return_std=True)
    sigma = np.maximum(sigma, 1e-9)
    improvement = y_best - mu - xi
    from scipy.stats import norm
    Z = improvement / sigma
    ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)
    ei[y_best - mu <= xi] = 0.0
    return ei

def bayesian_optimization(n_initial=10, n_iter=20, seed=42):
    """Simple Bayesian optimization loop."""
    rng = np.random.RandomState(seed)
    
    # Phase 1: random initialization
    X = rng.uniform(0, 1, size=(n_initial, 2))
    y = np.array([fake_cfd(p) for p in X])
    
    history = [(X.copy(), y.copy())]
    
    # Phase 2: BO loop
    for i in range(n_iter):
        # Fit GP surrogate
        kernel = Matern(nu=2.5, length_scale=[0.3, 0.3])
        gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=5, random_state=seed)
        gp.fit(X, y)
        
        # Evaluate acquisition function on a grid
        X_acq = np.linspace(0, 1, 200).reshape(-1, 1)
        X_acq = np.column_stack([X_acq[:, 0], np.linspace(0, 1, 200)])
        
        y_best = np.min(y)
        ei = expected_improvement(X_acq, gp, y_best)
        
        # Pick next point
        x_next = X_acq[np.argmax(ei)]
        
        # Evaluate
        y_next = fake_cfd(x_next)
        
        # Update
        X = np.vstack([X, x_next])
        y = np.append(y, y_next)
        
        history.append((X.copy(), y.copy()))
        
        if (i + 1) % 5 == 0:
            print(f'  Iter {i+1:2d}: best Cd = {np.min(y):.4f}')
    
    best_idx = np.argmin(y)
    return X, y, X[best_idx], y[best_idx], history

print('Running Bayesian optimization (30 evaluations total)...')
X_bo, y_bo, best_pt_bo, best_val_bo, history_bo = bayesian_optimization()
print(f'\nBO best: Cd = {best_val_bo:.4f} at {best_pt_bo}')

## 4. Comparison: BO vs Random Search

**Concept**: The key metric is **regret**: how far is your best found value from the true optimum? BO should have lower regret with the same budget.


In [ ]:
# True optimum (from the landscape)
true_opt = 0.010  # approximate global minimum

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: convergence
ax = axes[0]
# Random search convergence
rng = np.random.RandomState(42)
pts = rng.uniform(0, 1, size=(30, 2))
vals = np.array([fake_cfd(p) for p in pts])
best_so_far_rs = np.minimum.accumulate(vals)
ax.plot(range(1, 31), best_so_far_rs, 'r-o', markersize=3, label='Random search')

# BO convergence
best_so_far_bo = np.minimum.accumulate(y_bo)
ax.plot(range(1, 31), best_so_far_bo, 'b-s', markersize=3, label='Bayesian opt.')
ax.axhline(true_opt, color='gray', linestyle='--', alpha=0.5, label='True optimum')
ax.set_xlabel('Evaluations')
ax.set_ylabel('Best Cd found')
ax.set_title('Convergence: Best Value vs Evaluations')
ax.legend()
ax.grid(True, alpha=0.3)

# Right: final points
ax = axes[1]
contour = ax.contourf(X1, X2, Z, levels=30, cmap='viridis', alpha=0.6)
ax.scatter(pts[:, 0], pts[:, 1], c='red', s=30, alpha=0.5, label='Random')
ax.scatter(X_bo[:, 0], X_bo[:, 1], c='blue', s=30, alpha=0.5, label='BO')
ax.scatter(best_pt_rs[0], best_pt_rs[1], c='red', s=150, marker='*', label=f'RS best: {best_val_rs:.4f}')
ax.scatter(best_pt_bo[0], best_pt_bo[1], c='blue', s=150, marker='*', label=f'BO best: {best_val_bo:.4f}')
ax.set_xlabel('Param 1')
ax.set_ylabel('Param 2')
ax.set_title('Evaluation Points: Random vs BO')
ax.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

print(f'Random search: best Cd = {best_val_rs:.4f} (regret: {best_val_rs - true_opt:.4f})')
print(f'Bayesian opt:  best Cd = {best_val_bo:.4f} (regret: {best_val_bo - true_opt:.4f})')
print(f'BO improvement: {(best_val_rs - best_val_bo) / (best_val_rs - true_opt) * 100:.0f}% less regret')

## 5. Using piglot (If Installed)

**Concept**: piglot wraps these concepts in a clean API. You define the objective, the search space, and the budget — it handles the GP, acquisition function, and parallel evaluation.


In [ ]:
if HAS_PIGLOT:
    # piglot API (simplified)
    # In practice:
    #   import piglot
    #   config = piglot.Config(
    #       objective='my_objective',
    #       n_initial=10,
    #       n_iter=20,
    #       acquisition='ei',
    #       parallel=4
    #   )
    #   result = piglot.optimize(config)
    #   print(result.best_x, result.best_y)
    print('piglot is installed. See the piglot docs for the full API:')
    print('https://github.com/piglot-dev/piglot')
else:
    print('piglot not installed. The manual implementation above')
    print('shows the same logic piglot uses internally:')
    print()
    print('  1. Gaussian Process surrogate (sklearn.gaussian_process)')
    print('  2. Expected Improvement acquisition function')
    print('  3. Grid search for the next point')
    print('  4. Update and repeat')
    print()
    print('piglot adds: parallel evaluation, multi-objective,')
    print('constraint handling, and a cleaner API.')

## Engineering Takeaway

**When to use Bayesian optimization (piglot)**:
- Each evaluation is **expensive** (CFD: hours, FEA: minutes, experiment: days)
- The design space is **low-dimensional** (< 20 parameters)
- You have a **small budget** (30-200 evaluations)
- The objective is a **black box** (no gradients)

**When NOT to use it**:
- Evaluations are cheap (< 1s) -> use scipy.optimize or random search
- Design space is high-dimensional (> 20) -> use CMA-ES or evolutionary algorithms
- You have gradients -> use gradient-based methods (much faster)

**The 4-hour CFD problem**: 30 BO evaluations = 120 hours of compute. Random search with 30 evaluations = same compute, but 2-3x worse result. BO is the right tool.


## Exercises

1. **3D problem**: Extend the fake CFD to 3 parameters (camber, thickness, twist). Run BO with 50 evaluations. Does BO still beat random search in 3D?

2. **Multi-objective**: Modify the objective to return (Cd, Cl). Use a Pareto front approach (e.g., NSGA-II from DEAP) instead of BO. Compare the trade-off curves.

3. **Constraints**: Add a constraint: thickness > 0.08. How do you handle this in BO? (Hint: penalize infeasible points in the objective.)

4. **Real CFD**: Replace `fake_cfd` with a call to your OpenFOAM case (from the OpenFOAM notebook). Run BO with 10 evaluations. How close is the result to the true optimum?

5. **Parallel BO**: Modify the BO loop to evaluate 3 points in parallel (using `multiprocessing`). Does this speed up the wall-clock time? (Hint: the GP update is sequential, but the evaluations are parallel.)
